# 14 · Bayesian inversion — worked solutions

These solutions are most useful after attempting the chapter exercises. Each
section states the reasoning before the calculation; numerical values depend on
the compact, fixed-seed setup below.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

rng = np.random.default_rng(0)
fault = geodef.Fault.planar(
    lat=34.0, lon=-118.0, depth=9_000.0, strike=90.0, dip=30.0,
    length=24_000.0, width=12_000.0, n_length=4, n_width=3,
)
i = np.arange(fault.n_patches) % 4
j = np.arange(fault.n_patches) // 4
true_slip = np.exp(-((i - 1.5) / 1.2) ** 2 - ((j - 1.0) / 0.9) ** 2)
station_lon, station_lat = np.meshgrid(
    np.linspace(-118.18, -117.82, 6), np.linspace(33.90, 34.12, 5)
)
station_lon, station_lat = station_lon.ravel(), station_lat.ravel()
east, north, up = fault.displacement(
    station_lat, station_lon, slip_strike=0.0, slip_dip=true_slip
)
sigma = 0.004
gnss = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=east + rng.normal(0.0, sigma, east.size),
    north=north + rng.normal(0.0, sigma, north.size),
    up=up + rng.normal(0.0, sigma, up.size),
    sigma_east=sigma, sigma_north=sigma, sigma_up=sigma,
    name="solution_gnss",
)


## 1. Analytic correspondence

Without positivity, fixed-geometry Gaussian posterior mean equals the
regularized solution. Compute both under identical covariance and operator.

## 2–3. Positivity and prior scale

Positivity makes intervals asymmetric near zero. Prior-scale sensitivity should
be summarized in scientific quantities, not only parameter traces.

## 4–5. Chains and predictive checks

Multiple dispersed chains reveal missed modes that one long chain cannot.
Construct held-out stations where wrong geometries differ most; in-sample PPC
can pass when flexible slip compensates.


In [ ]:
linear = geodef.solve(
    fault, datasets=gnss, components="dip", regularization="damping",
    regularization_strength=1.0,
)
Cm = geodef.invert.model_covariance(linear, fault, gnss)
draws = rng.multivariate_normal(linear.dip_slip, Cm, size=2_000)
assert np.allclose(draws.mean(axis=0), linear.dip_slip, atol=0.03)
sensitivity = {}
for strength in (0.25, 1.0, 4.0):
    result = geodef.solve(
        fault, datasets=gnss, components="dip", regularization="damping",
        regularization_strength=strength,
    )
    sensitivity[strength] = (result.dip_slip.max(), fault.magnitude(np.abs(result.dip_slip)))
print("analytic draw mean:", draws.mean(axis=0))
print("prior-strength sensitivity (peak, Mw):", sensitivity)


## Interpretation and alternatives

This analytic check isolates the probability algebra. The chapter's NUTS workflow is required once positivity, hyperparameters, or geometry make the posterior non-Gaussian.
